# 2. Monte Carlo risk analysis and what-if experiments

The original project reported **one** simulation per scenario. Evacuation time is a random variable (pre-movement times, occupant mix and occupancy of open spaces are random), so a single run can understate the risk. Here we run 1,000 replications per scenario, look at the tail of the distribution (P95) and test the improvements proposed in the report quantitatively.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evacuation import *
from evacuation import config
from evacuation.plots import use_style, BLUE, ORANGE, AQUA, INK, INK_2, MUTED

use_style()
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
RES = Path("../results"); RES.mkdir(exist_ok=True)
pd.set_option("display.precision", 1)

## 2.1 Distribution of the total evacuation time

In [2]:
N = 1000
scen = ["E1", "E2", "E3", "E4"]
mc = {s: monte_carlo(s, n_reps=N) for s in scen}
summary = pd.DataFrame({s: summarize_tmax(mc[s]["runs"]) for s in scen}).T
summary.insert(0, "occupants (mean)", [mc[s]["runs"].people.mean() for s in scen])
summary.to_csv(RES / "tmax_by_scenario.csv"); summary

,occupants (mean),mean,std,p05,median,p95,max
E1,817.3,366.7,13.7,345.0,366.7,388.9,410.4
E2,640.4,340.5,8.3,326.6,340.5,353.8,364.8
E3,357.1,264.8,6.3,255.1,264.7,275.6,287.3
E4,268.0,200.1,4.7,192.8,199.9,208.4,218.3


In [3]:
fig, ax = plt.subplots(figsize=(8, 3.6))
data = [mc[s]["runs"].t_max for s in scen][::-1]
bp = ax.boxplot(data, vert=False, widths=0.5, whis=(5, 95), showfliers=False, patch_artist=True,
                medianprops=dict(color="white", lw=2), boxprops=dict(facecolor=BLUE, edgecolor=BLUE),
                whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE))
ax.set_yticks(range(1, 5), [f"{s}  {config.SCENARIOS[s]['label']}" for s in scen][::-1])
for i, s in enumerate(scen[::-1], start=1):
    p95 = summary.loc[s, "p95"]
    ax.annotate(f"P95 {p95:.0f} s", (p95, i), xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2)
ax.set_xlabel("Total evacuation time T_max (s)"); ax.grid(axis="y", visible=False)
ax.set_title(f"T_max by scenario ({N} replications; box = IQR, whiskers = P05-P95)")
fig.tight_layout(); fig.savefig(FIG / "02_tmax_by_scenario.png"); plt.show()

/tmp/ipykernel_761/878768484.py:3: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  bp = ax.boxplot(data, vert=False, widths=0.5, whis=(5, 95), showfliers=False, patch_artist=True,


## 2.2 Where do people wait? Bottleneck analysis (E1)

Mean queueing time per element, averaged over replications.

In [4]:
el = mc["E1"]["elements"].head(8)
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(el))[::-1]
colors = [ORANGE if n.startswith("ext_stair") else BLUE for n in el.index]
ax.barh(y, el.mean_wait, color=colors, height=0.6)
for yi, (n, w) in zip(y, el.mean_wait.items()):
    ax.annotate(f"{w:.0f} s", (w, yi), xytext=(4, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(y, el.index); ax.set_xlabel("Mean queueing time per person (s)"); ax.grid(axis="y", visible=False)
ax.set_title("E1: mean wait at the 8 most congested elements (orange = external stairs)")
fig.tight_layout(); fig.savefig(FIG / "02_bottlenecks_E1.png"); plt.show()
mc["E1"]["elements"].head(8)

,served,mean_wait,max_wait
element,,,
ext_stair_left_p1_ground,149.5,78.3,162.7
pm1_exit_left,90.0,57.5,110.6
pm1_exit_right,90.0,57.5,110.5
ext_stair_right_p1_ground,111.8,54.8,103.0
p0_right_door_inner,78.0,47.8,92.2
p1_door_left,78.0,47.8,92.1
p0_left_door_inner,78.0,47.7,92.2
p1_door_right,78.0,47.7,92.2


The left external stair to the ground has the longest queues **and** determines $T_{max}$: it is the only route that merges two floors (first-floor walkway + second floor). The basement doors also queue, but their flow ends after ~175 s, long before the stairs.

## 2.3 Testing the improvements proposed in the report

* **Reroute**: send the two outer medium classrooms of P1 to the interior stair (towards the landing exit) instead of the external walkway.
* **Widen**: add capacity to the external stair flights from P1 to the ground (5 -> 8 people simultaneously, i.e. ~60 % wider).
* **Both** combined.

All policies use the same seeds (common random numbers), so the differences are due to the policy, not to noise.

In [5]:
policies = {
    "Baseline": Building(),
    "Reroute P1 outer rooms": reroute_p1_mediums(),
    "Widen ground flights": widen_external_stairs(3),
    "Reroute + widen": reroute_p1_mediums(widen_external_stairs(3)),
}
pol = {k: monte_carlo("E1", n_reps=N, building=b) for k, b in policies.items()}
crit_exits = ["pm1_left", "pm1_right", "landing", "p0_left", "p0_center", "p0_right", "stair_ext_left", "stair_ext_right"]
rows = {}
for k, res in pol.items():
    s = summarize_tmax(res["runs"])
    clear = pd.Series({e: res["runs"][f"clear_{e}"].mean() for e in crit_exits})
    rows[k] = {"mean T_max": s["mean"], "P95 T_max": s["p95"], "std": s["std"],
               "critical exit": config.EXIT_LABELS[clear.idxmax()]}
interv = pd.DataFrame(rows).T
base = interv.loc["Baseline", "P95 T_max"]
interv["P95 reduction"] = (1 - interv["P95 T_max"].astype(float) / base).map("{:.0%}".format)
interv.to_csv(RES / "interventions_E1.csv"); interv

,mean T_max,P95 T_max,std,critical exit,P95 reduction
Baseline,366.7,388.9,13.7,External stair (left),0%
Reroute P1 outer rooms,301.8,322.8,13.0,External stair (left),17%
Widen ground flights,247.5,261.2,8.6,External stair (left),33%
Reroute + widen,228.4,235.3,4.1,Landing door,39%


In [6]:
fig, ax = plt.subplots(figsize=(8, 3.6))
names = list(pol)[::-1]
data = [pol[k]["runs"].t_max for k in names]
ax.boxplot(data, vert=False, widths=0.5, whis=(5, 95), showfliers=False, patch_artist=True,
           medianprops=dict(color="white", lw=2), boxprops=dict(facecolor=BLUE, edgecolor=BLUE),
           whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE))
ax.set_yticks(range(1, len(names) + 1), names)
for i, k in enumerate(names, start=1):
    p95 = interv.loc[k, "P95 T_max"]
    ax.annotate(f"P95 {p95:.0f} s", (p95, i), xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2)
ax.set_xlabel("Total evacuation time T_max (s), scenario E1"); ax.grid(axis="y", visible=False)
ax.set_title(f"Effect of the proposed improvements (E1, {N} replications each)")
fig.tight_layout(); fig.savefig(FIG / "02_interventions_E1.png"); plt.show()

/tmp/ipykernel_761/1268111826.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(data, vert=False, widths=0.5, whis=(5, 95), showfliers=False, patch_artist=True,


## 2.4 How much wider should the stairs be? Capacity sweep

Increasing the capacity of the ground flights reduces $T_{max}$ only until another element becomes the limit. Beyond ~10 people per flight the stairs no longer queue: the last person is now held back upstream, by the single doors from the first and second floors onto the walkway.

In [7]:
caps = [3, 4, 5, 6, 7, 8, 9, 10, 12]
sweep = []
for k in caps:
    b = Building()
    for n in b.elements:
        if n.endswith("p1_ground"):
            b.elements[n]["capacity"] = k
    res = monte_carlo("E1", n_reps=300, building=b)
    s = summarize_tmax(res["runs"])
    clear = pd.Series({e: res["runs"][f"clear_{e}"].mean() for e in crit_exits})
    sweep.append({"capacity": k, "mean": s["mean"], "p05": s["p05"], "p95": s["p95"],
                  "critical exit": config.EXIT_LABELS[clear.idxmax()]})
sweep = pd.DataFrame(sweep).set_index("capacity"); sweep.to_csv(RES / "stair_capacity_sweep_E1.csv"); sweep

,mean,p05,p95,critical exit
capacity,,,,
3,582.5,542.3,621.4,External stair (left)
4,446.8,418.2,476.7,External stair (left)
5,366.3,342.8,389.2,External stair (left)
6,312.7,292.3,332.0,External stair (left)
7,275.1,257.5,291.8,External stair (left)
8,247.3,230.6,261.5,External stair (left)
9,226.3,212.3,238.8,External stair (left)
10,218.2,199.5,234.8,External stair (left)
12,217.5,192.9,234.8,External stair (left)


In [8]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.fill_between(sweep.index, sweep.p05, sweep.p95, color=BLUE, alpha=0.2, lw=0, label="P05-P95")
ax.plot(sweep.index, sweep["mean"], "-o", color=BLUE, ms=6, label="mean")
ax.axvline(5, color=MUTED, lw=1, ls="--")
ax.annotate("current", (5, ax.get_ylim()[1]), xytext=(4, -4), textcoords="offset points", va="top", fontsize=8.5, color=INK_2)
floor = sweep["mean"].iloc[-1]
ax.axhline(floor, color=ORANGE, lw=1, ls=":")
ax.annotate("floor: the single walkway doors upstream now limit the flow", (caps[-1], floor), xytext=(0, 6), textcoords="offset points",
            ha="right", fontsize=8.5, color=INK_2)
ax.set_xlabel("Capacity of each external stair flight to the ground (people)"); ax.set_ylabel("T_max (s)")
ax.set_title("E1: diminishing returns of widening the external stairs"); ax.legend(loc="upper right")
fig.tight_layout(); fig.savefig(FIG / "02_capacity_sweep_E1.png"); plt.show()

## 2.5 Conclusions

* At peak hours (E1) the evacuation takes about **6 minutes on average**, but in 1 out of 20 replications it exceeds the P95 shown above: a single deterministic run understates the tail.
* The **left external stair to the ground** is the critical element in every scenario, because it merges the flows of two floors.
* **Rerouting** costs nothing and already cuts the tail; **widening** the ground flights helps more; **both** together bring the critical exit back inside the building (landing door / basement doors), so further widening has diminishing returns.

**Limitations.** Door and stair dimensions are calibrated, not measured; occupants follow their assigned route with no individual interaction or route switching; no validation against drills or regulation. Results are comparative (policy A vs policy B) rather than absolute safety figures.